In [ ]:
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm

In [ ]:
def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y[:, np.newaxis] - y_pred)**2) / np.sum(y**2)
    return r2

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_grad(x):
    return sigmoid(x) * (1 - sigmoid(x))

def mean_squared_error(y, t):
    return 0.5 * np.sum((y - t) ** 2)

class ThreeLayerNet:

    def __init__(self, input_size, hidden_size1, hidden_size2, output_size, weight_init_std=0.01):
    # 初始化权重
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size1)
        self.params['b1'] = np.zeros(hidden_size1)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size1, hidden_size2)
        self.params['b2'] = np.zeros(hidden_size2)
        self.params['W3'] = weight_init_std * np.random.randn(hidden_size2, output_size)
        self.params['b3'] = np.zeros(output_size)
    
    def predict(self, x):
        W1, W2, W3 = self.params['W1'], self.params['W2'], self.params['W3']
        b1, b2, b3 = self.params['b1'], self.params['b2'], self.params['b3']
        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        z2 = sigmoid(a2)
        a3 = np.dot(z2, W3) + b3
        y = a3
        
        return y
        
    # x:输入数据, t:监督数据
    def loss(self, x, t):
        y = self.predict(x)
        return mean_squared_error(y, t)
    
    def accuracy(self, x, t):
        y = self.predict(x)
        error = np.mean((y - t[:, np.newaxis]) ** 2)  # 使用均方根误差(MSE)来衡量回归模型的性能
        
        return error
        
    # x:输入数据, t:监督数据   
    def gradient(self, x, t):
        W1, W2, W3 = self.params['W1'], self.params['W2'], self.params['W3']
        b1, b2, b3 = self.params['b1'], self.params['b2'], self.params['b3']
        grads = {}
        
        batch_num = x.shape[0]
        
        # forward
        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        z2 = sigmoid(a2)
        a3 = np.dot(z2, W3) + b3
        y = a3
  
        # backward
        dy = (y - t[:, np.newaxis]) / batch_num
        grads['W3'] = np.dot(z2.T, dy)
        grads['b3'] = np.sum(dy, axis=0)
        
        da2 = np.dot(dy, W3.T)
        dz2 = sigmoid_grad(a2) * da2
        grads['W2'] = np.dot(z1.T, dz2)
        grads['b2'] = np.sum(dz2, axis=0)
    
        da1 = np.dot(dz2, W2.T)
        dz1 = sigmoid_grad(a1) * da1
        grads['W1'] = np.dot(x.T, dz1)
        grads['b1'] = np.sum(dz1, axis=0)

        return grads

In [ ]:
with open("characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
data.head()

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

pred_y_list = []
true_y_list = []

# Set the parameter
n_epochs = 100
batch_size = 5000
learning_rate = 0.02

patience = 20
best_model_state = None
best_loss = float('inf')
best_R2 = float('-inf')

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training, validation, and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]

    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values 
    y_train = train_data['RET'].values
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_val = val_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values

    net_work = ThreeLayerNet(input_size=94, hidden_size1=16, hidden_size2=16, output_size=1)

    train_size = X_train.shape[0]
    counter = 0

    for epoch in range(1,n_epochs+1):
        learning_rate = learning_rate / 1.05
        for i in range(0, train_size, batch_size):
            xi = X_train[i:i + batch_size]
            yi = y_train[i:i + batch_size]

            grad = net_work.gradient(xi, yi)
            for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3'):
                net_work.params[key] -= learning_rate * grad[key]
    
        val_loss = net_work.accuracy(X_val, y_val)
    
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = net_work.params.copy()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

        if epoch % 30 == 0:
            train_loss = net_work.accuracy(X_train, y_train)
            test_loss = net_work.accuracy(X_val, y_val)
            print(f"train loss: {train_loss}, val loss {val_loss}")
        if epoch % 10 == 0:
            print('已进行{}次,counter:{}'.format(epoch, counter))
            
    for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3'):
        net_work.params[key] = best_model_state[key]
    y_pred = net_work.predict(X_test)
    print(f"train loss: {train_loss}, val loss {val_loss}")
    R2 = R2_cal(y_test, y_pred)*100
    print(R2)
    if R2 > best_R2:
        best_R2 = R2

In [ ]:
best_R2

PartⅡ：筛选重要特征值

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

# Set the parameter
n_epochs = 100
batch_size = 5000
learning_rate = 0.02

patience = 20
best_model_state = None
best_loss = float('inf')

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training, validation, and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]

    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values 
    y_train = train_data['RET'].values
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_val = val_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values

    net_work = ThreeLayerNet(input_size=94, hidden_size1=16, hidden_size2=16, output_size=1)

    train_size = X_train.shape[0]
    counter = 0

    for epoch in range(1,n_epochs+1):
        learning_rate = learning_rate / 1.05
        for i in range(0, train_size, batch_size):
            xi = X_train[i:i + batch_size]
            yi = y_train[i:i + batch_size]

            grad = net_work.gradient(xi, yi)
            for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3'):
                net_work.params[key] -= learning_rate * grad[key]
    
        val_loss = net_work.accuracy(X_val, y_val)
    
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = net_work.params.copy()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

        if epoch % 30 == 0:
            train_loss = net_work.accuracy(X_train, y_train)
            test_loss = net_work.accuracy(X_val, y_val)
            print(f"train loss: {train_loss}, val loss {val_loss}")
        if epoch % 10 == 0:
            print('已进行{}次,counter:{}'.format(epoch, counter))
            
    for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3'):
        net_work.params[key] = best_model_state[key]
    y_pred = net_work.predict(X_test)
    print(f"train loss: {train_loss}, val loss {val_loss}")
    R2 = R2_cal(y_test, y_pred)*100
    print(f'origional R2:{R2}')

    
    r2_differences = {}
    feature_names = ['mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn', 'absacc', 'acc', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep', 'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'invest', 'lev', 'lgr', 'mve_ia', 'operprof', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr', 'pchgm_pchsale', 'pchquick', 'pchsale_pchinvt', 'pchsale_pchrect', 'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'ps', 'quick', 'rd', 'rd_mve', 'rd_sale', 'realestate', 'roic', 'salecash', 'saleinv', 'salerec', 'secured', 'securedind', 'sgr', 'sin', 'sp', 'tang', 'tb', 'aeavol', 'cash', 'chtx', 'cinvest', 'ear', 'nincr', 'roaq', 'roavol', 'roeq', 'rsup', 'stdacc', 'stdcf', 'ms', 'baspread', 'ill', 'maxret', 'retvol', 'std_dolvol', 'std_turn', 'zerotrade']
    learning_rate_m = 0.02
    best_model_state_m = None
    best_loss_m = float('inf')
    
    for name in feature_names:
        # Create copy
        X_train_copy = X_train.copy()
        X_val_copy = X_val.copy()
        X_test_copy = X_test.copy()

        index_ = feature_names.index(name)
        X_train_copy[index_] = 0
        X_val_copy[index_] = 0
        X_test_copy[index_] = 0

        net_work_m = ThreeLayerNet(input_size=94, hidden_size1=16, hidden_size2=16, output_size=1)

        train_size_m = X_train_copy.shape[0]
        counter_m = 0
    
        for epoch in range(1,n_epochs+1):
            learning_rate_m = learning_rate_m / 1.05
            for i in range(0, train_size_m, batch_size):
                xi_m = X_train_copy[i:i + batch_size]
                yi_m = y_train[i:i + batch_size]
    
                grad_m = net_work_m.gradient(xi_m, yi_m)
                for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3'):
                    net_work_m.params[key] -= learning_rate_m * grad_m[key]
        
            val_loss_m = net_work_m.accuracy(X_val_copy, y_val)
        
            if val_loss_m < best_loss_m:
                best_loss_m = val_loss_m
                best_model_state_m = net_work_m.params.copy()
                counter_m = 0
            else:
                counter_m += 1
                if counter_m >= patience:
                    break
    
        for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3'):
            net_work_m.params[key] = best_model_state_m[key]
        y_pred_m = net_work_m.predict(X_test_copy)
        R2_m = R2_cal(y_test, y_pred_m)*100
        r2_differences[name] = R2 - R2_m
        if (index_+1) % 10 == 0:
            print(f'{index_+1}features have been completed')

    sorted_dict = sorted(r2_differences.items(), key=lambda item: item[1], reverse=True)
    print(f'top five important features:{sorted_dict[:5]}')

In [ ]:
sorted_dict = sorted(r2_differences.items(), key=lambda item: item[1], reverse=True)
print(f'top five important features:{sorted_dict[:5]}')